In [4]:
!unzip pretrain.zip

Archive:  pretrain.zip
   creating: pretrain/
  inflating: pretrain/test_encyclopedia.json  
  inflating: pretrain/train_encyclopedia.json  


In [5]:
!pip install datasets

Looking in indexes: http://mirrors.aliyun.com/pypi/simple


In [15]:
import torch
import os

os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

In [12]:
from datasets import load_dataset
from transformers import default_data_collator

dataset = load_dataset('json', data_files='pretrain/train_encyclopedia.json',split = 'train')
print(dataset)

def f(data):
    data = [i['text'] for i in data]

    data = tokenizer(data,
                     padding=True,
                     truncation=True,
                     max_length=300,
                     return_tensors='pt').to(device)

    data['labels'] = data['input_ids'].clone()
    select = data['labels'] == tokenizer.pad_token_id
    data['labels'][select] = -100

    return data


loader = torch.utils.data.DataLoader(dataset,
                                     batch_size=4,
                                     shuffle=True,
                                     drop_last=True,
                                     collate_fn=f)

len(loader), next(iter(loader))

Dataset({
    features: ['text'],
    num_rows: 361420
})


(90355,
 {'input_ids': tensor([[32003,   223, 33176,  ...,   162,   232,   246],
         [  165,   109,   120,  ...,   227, 32849,   121],
         [30298,   235, 26344,  ...,   245, 16764,   165],
         [20998,   234, 20998,  ...,   242,   117, 20998]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1]], device='cuda:0'), 'labels': tensor([[32003,   223, 33176,  ...,   162,   232,   246],
         [  165,   109,   120,  ...,   227, 32849,   121],
         [30298,   235, 26344,  ...,   245, 16764,   165],
         [20998,   234, 20998,  ...,   242,   117, 20998]], device='cuda:0')})

In [13]:
from transformers import AutoModelForCausalLM

model_actor = AutoModelForCausalLM.from_pretrained('gpt2').to(
    device)

model_actor.config


GPT2Config {
  "_attn_implementation_autoset": true,
  "_name_or_path": "gpt2",
  "activation_function": "gelu_new",
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
    }
  },
  "transformers_version": "4.48.0",
  "use_cache": true,
  "vocab_size": 50257
}

In [14]:
optimizer = torch.optim.Adam(model_actor.parameters(), lr=1e-5)

for i, data in enumerate(loader):
    out = model_actor(**data)
    out.loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    if i % 1000 == 0:
        print(i, len(loader), out.loss.item())

        prompt = data['input_ids'][0]
        chosen = prompt[50:]
        prompt = prompt[:50]

        gen = model_actor.generate(prompt.unsqueeze(0),
                                   max_length=300,
                                   pad_token_id=tokenizer.pad_token_id,
                                   eos_token_id=tokenizer.eos_token_id)[0, 50:]

        print('prompt=', tokenizer.decode(prompt))
        print('chosen=', tokenizer.decode(chosen))
        print('gen=', tokenizer.decode(gen))

model_actor.save_pretrained('model/actor')

0 90355 2.906944751739502
prompt= 心病毒属是什么?？心病毒属，又称为心脏病�
chosen= ��属，是微小核糖核酸病毒科里的一属，脊椎动物是这类病毒的天然宿主，下含甲型心病毒至己型心病毒6个物种，最初于1981年得到培养分离。心病毒属的病毒都是单链RNA病毒、无病毒包膜包膜包裹，直径约30纳米、基因组长约7.8千碱基。<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>
gen= ��属?？心病毒属，又称为心脏病毒属?？心病毒属，又称为心脏病毒属?？心病毒属，又称为心脏病毒属?？心病毒属，又称为心脏病毒属?？心病毒属，又称为心脏病毒属?？心病毒属，又称为心脏病毒属?？心病�
1000 90355 2.1878626346588135
prompt= 虚烦失眠的治疗和预防方法？治疗方法：基本方
chosen= ：熟地15g，茯苓20g，枣皮10g，泽泻10g，丹皮10g，淮山药20g，酸枣仁20g，知母10g，川芎10g，甘草10g。用法：将以上药物纳入药锅中，温水泡之30分钟，以文火煮沸20分钟，取药汁200ml ，温服，每天2次，2日剂。用药期间停服镇静药，晚饭后�
gen= 法，治疗方法，治疗方法，治疗方法，治疗方法，治疗方法，治疗方法，治疗方法，治疗方法，治疗方法，治疗方法，治疗方法，治疗方法，治疗方法，治疗方法，治疗方法，治疗方法，治疗方法，治疗方法，治疗方法，治疗方法，治�
2000 90355 2.1712915897369385
prompt= 利普刀后宫颈管粘连的症状？宫颈管粘连是指�
chosen= �于宫颈管粘膜受机械损伤后粘连致使颈管狭窄或闭锁。子宫颈管粘连偶见于人工流产后或宫颈电烙术后，这类手术后出现月经过少伴痛经或闭经伴发周期性严重腹痛时应考虑本病。可用进行粘连剥离后进行沙棘油纱布进行填充治疗，同时应用�
gen= �于宫颈管粘连的症状，宫颈管粘连是指由于宫颈管粘连的症状，宫颈管粘连是指由于宫颈管粘连的症状，宫颈管粘连是指由于宫颈管粘连的症状，宫颈管粘连是指由于宫颈管粘连的症状，宫颈管粘连是指由于宫颈管粘连的症状，宫�
3000 90355 1.7830623388290405